# Agent Architectures: ReAct, Plan-and-Execute and Reflection, Compared

The full code of [Agent Architectures: ReAct, Plan-and-Execute and Reflection, Compared](https://residual-stream.lvtan71.workers.dev/posts/agent-architectures-react-plan-reflection/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/agent-architectures-react-plan-reflection) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/agent-architectures-react-plan-reflection.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 33 min end to end on the RTX 3090 machine that saved the outputs below.
The agents sample, so scores move between machines; on an RTX 3090, ReAct came second and reflection third.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 33 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "torch==2.14.*" "transformers==5.18.*" huggingface_hub numpy

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "agent-architectures-react-plan-reflection"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/agent-architectures-react-plan-reflection


## `arch.py`

Twelve fictional companies, `lookup` and `calculator` tools, and 20 questions; ReAct, plan-and-execute and reflection run each question three times with Qwen2.5-3B-Instruct at temperature 0.7. Prints accuracy, tokens and seconds per question, and logs every trial to `arch_trials_v3.jsonl` (a stopped run resumes from it).

In [2]:
!{sys.executable} arch.py

20 questions x 3 runs; Qwen2.5-3B-Instruct

architecture        right  tokens processed/question  seconds/question


ReAct                 42%                      1,195               4.4


plan-and-execute      58%                      5,816              17.6


reflection            35%                      2,540              10.8


The post printed:

```text
20 questions x 3 runs; Qwen2.5-3B-Instruct

architecture        right  tokens processed/question  seconds/question
ReAct                 35%                      1,176              12.1
plan-and-execute      48%                      6,058              51.7
reflection            37%                      2,420              29.6
```

## `failures.py`

Rebuilds the same questions and sorts every logged trial: right, sign flipped, a total missing one company, no answer, other; accuracy by question type; paired comparisons; and what reflection's critic flagged and what its retries changed; then the counts the post quotes from the replies (sign flips, failed lookups the reply mentions, what the critic said, unanswered retries). Reads `arch_trials_v3.jsonl`, or the log named as its argument.

In [3]:
!{sys.executable} failures.py

                       right [95% CI]  sign flipped  total missing one  no answer  other
ReAct                  42% [30%, 54%]            21                  6          0      8
plan-and-execute       58% [46%, 70%]            21                  2          0      2
reflection             35% [24%, 48%]            20                  4          2     13

right, by question type:
  ReAct              years older 1/21, more employees 16/21, total of three 8/18
  plan-and-execute   years older 1/21, more employees 20/21, total of three 14/18
  reflection         years older 1/21, more employees 12/21, total of three 8/18

paired (same question and run): only the first right / only the second right
  ReAct vs plan-and-execute: 4 / 14
  ReAct vs reflection: 6 / 2
  plan-and-execute vs reflection: 17 / 3

reflection: the critic flagged 40 of 60 first attempts (19 of the 25 right ones, 21 of the 35 wrong ones)
  after the retry: wrong->right 2, right->wrong 6, wrong->wrong 19, right->right 13

The post printed:

```text
                       right [95% CI]  sign flipped  total missing one  no answer  other
ReAct                  35% [24%, 48%]            17                  9          1     12
reflection             37% [26%, 49%]            19                  4          8      7
plan-and-execute       48% [36%, 61%]            22                  2          0      7

right, by question type:
  ReAct              years older 1/21, more employees 16/21, total of three 4/18
  reflection         years older 2/21, more employees 13/21, total of three 7/18
  plan-and-execute   years older 0/21, more employees 18/21, total of three 11/18

paired (same question and run): only the first right / only the second right
  ReAct vs reflection: 5 / 6
  ReAct vs plan-and-execute: 4 / 12
  reflection vs plan-and-execute: 5 / 12

reflection: the critic flagged 44 of 60 first attempts (16 of the 21 right ones, 28 of the 39 wrong ones)
  after the retry: wrong->right 6, right->wrong 5, wrong->wrong 22, right->right 11
  first attempts with the sign flipped: 17; flagged 11; right after the retry 1

years older, wrong: 60; the right number with the wrong sign 51
ReAct, wrong totals: 14; the reply says a lookup failed 8
reflection: critic replies other than OK 44; first sentence calls it correct 42
reflection: unanswered 8; after a retry 7
```